# Lab 02 - Advanced Vision Tasks

In this lab, we will cover:
1. Image Classification (CIFAR-10)
2. Object Detection (Faster R-CNN)
3. Image Segmentation (DeepLabV3)

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import requests
from io import BytesIO

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Part 1: Image Classification (CIFAR-10)
We will train a simple CNN on the CIFAR-10 dataset.

### Question 1.1: Data Loading

Complete the code below to:
1. Define the hyperparameters (BATCH_SIZE=128, NUM_EPOCHS=5, LEARNING_RATE=0.001)
2. Create a transform pipeline that converts images to tensors and normalizes them with mean and std of 0.5 for all channels
3. Load the CIFAR-10 training and test datasets
4. Create DataLoaders for both datasets

In [ ]:
# TODO: Define hyperparameters
BATCH_SIZE = # YOUR CODE HERE
NUM_EPOCHS = # YOUR CODE HERE
LEARNING_RATE = # YOUR CODE HERE

# TODO: Define transformations (ToTensor and Normalize with (0.5, 0.5, 0.5) for mean and std)
transform = # YOUR CODE HERE

# TODO: Load CIFAR-10 datasets (use root='./data', download=True)
train_dataset = # YOUR CODE HERE
test_dataset = # YOUR CODE HERE

# TODO: Create DataLoaders (shuffle=True for train, shuffle=False for test)
train_loader = # YOUR CODE HERE
test_loader = # YOUR CODE HERE

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

### Question 1.2: CNN Model Architecture

Complete the SimpleCNN class below. The architecture should have:
- `conv1`: Conv2d layer with 3 input channels, 32 output channels, kernel size 3, padding 1
- `pool`: MaxPool2d with kernel size 2 and stride 2
- `conv2`: Conv2d layer with 32 input channels, 64 output channels, kernel size 3, padding 1
- `fc1`: Linear layer with input size 64*8*8 and output size 512
- `fc2`: Linear layer with input size 512 and output size 10 (number of classes)
- `relu`: ReLU activation

In the forward pass:
1. Apply conv1 -> relu -> pool
2. Apply conv2 -> relu -> pool
3. Flatten the tensor
4. Apply fc1 -> relu
5. Apply fc2 and return

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # TODO: Define the layers
        self.conv1 = # YOUR CODE HERE
        self.pool = # YOUR CODE HERE
        self.conv2 = # YOUR CODE HERE
        self.fc1 = # YOUR CODE HERE
        self.fc2 = # YOUR CODE HERE
        self.relu = # YOUR CODE HERE

    def forward(self, x):
        # TODO: Implement the forward pass
        # YOUR CODE HERE
        pass

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

### Question 1.3: Training Loop

Complete the training loop below. For each epoch and batch:
1. Move images and labels to the device
2. Perform forward pass to get outputs
3. Calculate the loss using criterion
4. Zero the gradients
5. Perform backward pass
6. Update the weights using optimizer

In [ ]:
for epoch in range(NUM_EPOCHS):
    for i, (images, labels) in enumerate(train_loader):
        # TODO: Move data to device
        images = # YOUR CODE HERE
        labels = # YOUR CODE HERE

        # TODO: Forward pass
        outputs = # YOUR CODE HERE
        loss = # YOUR CODE HERE

        # TODO: Backward and optimize
        # YOUR CODE HERE (3 lines: zero_grad, backward, step)

        if (i+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

### Question 1.4: Model Evaluation

Complete the evaluation code below to:
1. Set the model to evaluation mode
2. Disable gradient computation
3. Calculate the accuracy on the test set

In [ ]:
# TODO: Set model to evaluation mode
# YOUR CODE HERE

with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # TODO: Get model predictions
        outputs = # YOUR CODE HERE
        _, predicted = torch.max(outputs.data, 1)
        
        # TODO: Update total and correct counts
        total += # YOUR CODE HERE
        correct += # YOUR CODE HERE

    print(f'Accuracy of the model on the 10000 test images: {100 * correct / total} %')

## Part 2: Object Detection
We will use a pre-trained Faster R-CNN model.

### Question 2.1: Loading Pre-trained Model and Running Inference

Complete the code below to:
1. Load a pre-trained Faster R-CNN model with ResNet-50 FPN backbone
2. Set the model to evaluation mode and move it to the device
3. Load and display the sample image from 'data/cats.jpg'
4. Transform the image to a tensor and run inference

In [ ]:
# TODO: Load pre-trained Faster R-CNN model from torchvision (should be fasterrcnn_resnet50_fpn)
model_det = # YOUR CODE HERE

# TODO: Set model to eval mode and move to device
# YOUR CODE HERE

# Load sample image
img = 'data/cats.jpg'
image = Image.open(img).convert("RGB")
plt.imshow(image)
plt.show()

# TODO: Create transform and convert image to tensor
transform_det = # YOUR CODE HERE
img_tensor = # YOUR CODE HERE (don't forget unsqueeze and move to device)

# TODO: Run inference (remember to disable gradients)
# YOUR CODE HERE
prediction = # YOUR CODE HERE

### Question 2.2: Visualizing Detections

Complete the `plot_detection` function below to:
1. Extract boxes, scores, and labels from the prediction
2. For each box with score above threshold, draw a rectangle and display the score

In [ ]:
import matplotlib.patches as patches

def plot_detection(img, prediction, threshold=0.8):
    fig, ax = plt.subplots(1)
    ax.imshow(img)

    # TODO: Extract boxes, scores, and labels from prediction[0]
    # Remember to move tensors to CPU and convert to numpy
    boxes = # YOUR CODE HERE
    scores = # YOUR CODE HERE
    labels = # YOUR CODE HERE

    for i, box in enumerate(boxes):
        if scores[i] > threshold:
            # TODO: Create a Rectangle patch and add to axes
            # Rectangle takes (x, y), width, height as arguments
            # box format is [x1, y1, x2, y2]
            rect = # YOUR CODE HERE
            ax.add_patch(rect)
            plt.text(box[0], box[1], f'{scores[i]:.2f}', color='white', bbox=dict(facecolor='red', alpha=0.5))

    plt.show()

plot_detection(image, prediction)

## Part 3: YOLOv5 Person Detection on GIF

In this section, we will use YOLOv5 to detect people in a GIF animation.

**Important:** Run this code from within the `yolov5` folder.

### Question 3.1: YOLOv5 Person Detection

Complete the code below to:
1. Load the YOLOv5s model using torch.hub
2. Configure it to detect only people (class 0) with 25% confidence threshold
3. Process each frame of the GIF through the model
4. Annotate frames with detection results and save the output GIF

In [ ]:
import torch
import cv2
from PIL import Image
import numpy as np

# TODO: Load YOLOv5 model using torch.hub.load
# Use 'ultralytics/yolov5' as the repo and 'yolov5s' as the model
print("Loading YOLOv5 model...")
model = # YOUR CODE HERE

# TODO: Configure model to detect only people (class 0) with 25% confidence
model.classes = # YOUR CODE HERE
model.conf = # YOUR CODE HERE

# Load GIF (make sure walk.gif is in the data/ folder)
gif_path = "data/walk.gif"
gif = Image.open(gif_path)

# Extract frames from GIF
frames = []
try:
    while True:
        frames.append(np.array(gif.convert('RGB')))
        gif.seek(gif.tell() + 1)
except EOFError:
    pass

print(f"Processing {len(frames)} frames...")

# Process each frame
output_frames = []
for i, frame in enumerate(frames):
    # TODO: Run detection on frame
    results = # YOUR CODE HERE
    
    # TODO: Get annotated frame using results.render()
    annotated = # YOUR CODE HERE
    
    # Convert BGR to RGB for PIL
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    # TODO: Get number of people detected using results.pandas().xyxy[0]
    num_people = # YOUR CODE HERE
    
    # Add frame info text
    text = f"Frame {i+1}/{len(frames)} - People: {num_people}"
    cv2.putText(annotated_rgb, text, (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    
    output_frames.append(Image.fromarray(annotated_rgb))
    print(f"Frame {i+1}: {num_people} people detected")

# Save output GIF
print("Saving output...")
output_frames[0].save(
    'output_detected.gif',
    save_all=True,
    append_images=output_frames[1:],
    duration=gif.info.get('duration', 100),
    loop=0
)

print("Done! Output saved to: output_detected.gif")